# Historical absolute-pose baseline (exploratory, non-matched evaluation)

Second gate of the protocol: train the reference model, then check that
(i) E1a modes are **not trivially saturated** and (ii) E1b poses a **real orientation challenge**.

**Runtime → Change runtime type → GPU (T4 is enough).** Only `abs_pose` is trained here; `rel_pose`, `rel_inc` and `rfm_so3` come at later gates.

Read every TV5 number against the **oracle noise floor** printed next to it (a perfect model re-drawing GT at the same K does not score 0).

## 1. Code + environment

In [ ]:
import os, sys, glob, json, subprocess
REPO_URL = ""  # Optional Git remote URL for Colab; otherwise upload the repository ZIP.
IN_COLAB = "google.colab" in sys.modules

if not os.path.isdir("camera_motion"):
    candidates = sorted(os.path.dirname(p) for p in glob.glob("*/camera_motion") if os.path.isdir(p))
    if not candidates and IN_COLAB:
        if REPO_URL:
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "camera-motion-flow-matching"], check=True)
        else:
            from google.colab import files
            from zipfile import ZipFile
            uploaded = files.upload()
            if not uploaded:
                raise RuntimeError("Upload the repository ZIP or specify REPO_URL.")
            with ZipFile(next(iter(uploaded))) as zf:
                zf.extractall(".")
        candidates = sorted(os.path.dirname(p) for p in glob.glob("*/camera_motion") if os.path.isdir(p))
    if candidates:
        os.chdir(candidates[0])
assert os.path.isdir("camera_motion"), f"Cannot find repository root: {os.getcwd()}"
sys.path.insert(0, os.getcwd())
print("repository root:", os.getcwd())


RUNS = "runs"
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS = "/content/drive/MyDrive/camera_motion_runs"
os.makedirs(RUNS, exist_ok=True)

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY - switch runtime to GPU")
print("runs ->", os.path.abspath(RUNS))

## 2. Tests

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"], capture_output=True, text=True)
print(r.stdout[-800:]); assert r.returncode == 0, r.stderr[-1500:]

## 3. Train `fm_abs_pose` on E1a and E1b
Same architecture and budget for both families. Re-running a cell resumes from `ckpt_last.pt`.

In [ ]:
STEPS, N_TRAIN, BATCH, SEED = 20000, 10000, 256, 0
QUICK = False            # True: 2k steps to check the pipeline first
if QUICK: STEPS = 2000
RUN = {fam: f"{RUNS}/fm_abs_pose_{fam}_s{SEED}" for fam in ("E1a", "E1b")}

In [ ]:
!python -u scripts/train_fm.py --rep abs_pose --family E1a --n-train {N_TRAIN} --steps {STEPS} --batch {BATCH} --seed {SEED} --log-every 500 --out {RUN['E1a']}

In [ ]:
!python -u scripts/train_fm.py --rep abs_pose --family E1b --n-train {N_TRAIN} --steps {STEPS} --batch {BATCH} --seed {SEED} --log-every 500 --out {RUN['E1b']}

## 4. Evaluate (K = 200 samples per held-out condition, Euler, NFE = 50)

In [ ]:
!python -u scripts/eval_fm.py --run {RUN['E1a']} --K 200 --nfe 50

In [ ]:
!python -u scripts/eval_fm.py --run {RUN['E1b']} --K 200 --nfe 50

## 5. Gate-2 summary

In [ ]:
import numpy as np
def load(fam): return json.load(open(f"{RUN[fam]}/eval_m2_K200_nfe50_euler/eval_report.json"))
rows = []
for fam in ("E1a", "E1b"):
    rep = load(fam)
    for panel, rr in rep["panels"].items():
        r, m = rr["model"], rr["model"]["mode"]
        rows.append((f"{fam}/{panel}", m["tv5_mean"], rr["oracle"]["mode"]["tv5_mean"], m["coverage4_mean"],
                     m["full_coverage_rate"], m["unassigned_rate_mean"], m["minority_error_mean"],
                     r["endpoint"]["e_p_median"], r["endpoint"]["e_R_deg_median"]))
print(f"{'panel':28s} {'TV5':>6s} {'floor':>6s} {'cov':>5s} {'full':>5s} {'unasg':>6s} {'minErr':>7s} {'e_p':>6s} {'e_R':>6s}")
for r in rows:
    print(f"{r[0]:28s} {r[1]:6.3f} {r[2]:6.3f} {r[3]:5.2f} {r[4]:5.2f} {r[5]:6.3f} {r[6]:+7.3f} {r[7]:6.3f} {r[8]:6.2f}")
b = load("E1b")["panels"]["eval_id"]
print("\nE1b excursion W1 pooled  model :", {k: round(v, 3) for k, v in b["model"]["excursion_w1_pooled"].items()})
print("E1b excursion W1 pooled  oracle:", {k: round(v, 3) for k, v in b["oracle"]["excursion_w1_pooled"].items()})
for fam in ("E1a", "E1b"):
    s = json.load(open(f"{RUN[fam]}/train_summary.json")); m = json.load(open(f"{RUN[fam]}/run_meta.json"))
    print(f"{fam}: {m['params']:,} params | {s['steps_per_sec']} steps/s | peak mem {s.get('peak_mem_mb')} MB | {m['gpu']}")

**Reading the gate.** E1a is *saturated* if TV5 is within the oracle floor CI, coverage is 4/4 everywhere and unassigned ≈ 0 — then the protocol says make the data harder before adding models. E1b is a *real orientation challenge* if the excursion W1 is clearly above its noise floor or the excursion histograms visibly differ.

## 6. Figures

In [ ]:
from IPython.display import Image, display
for fam in ("E1a", "E1b"):
    for f in sorted(glob.glob(f"{RUN[fam]}/eval_m2_K200_nfe50_euler/figures/*.png")):
        print(fam, os.path.basename(f)); display(Image(filename=f))

## 7. Pack results to send back
Checkpoints are excluded by default (set `WITH_CKPT=True` to include).

In [ ]:
WITH_CKPT = False
import zipfile
with zipfile.ZipFile("phase2_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for fam in ("E1a", "E1b"):
        for f in glob.glob(f"{RUN[fam]}/**/*", recursive=True):
            if os.path.isdir(f) or (f.endswith(".pt") and not WITH_CKPT):
                continue
            z.write(f, os.path.relpath(f, RUNS))
print("phase2_results.zip", round(os.path.getsize("phase2_results.zip") / 2**20, 1), "MB")
if IN_COLAB:
    from google.colab import files; files.download("phase2_results.zip")